# Cuaderno de Creación y Simulación de Dataset IIoT



## Objetivos
1. Simular señales de telemetría realistas de tres sensores industriales clave (Temperatura, Vibración, Presión).
2. Modelar la **irregularidad temporal (muestreo asíncrono)** mediante intervalos variables $\Delta t$ siguiendo una distribución exponencial.
3. Inyectar firmas físicas de fallas típicas en la industria: picos en la vibración, sobrecalentamiento gradual y fugas de presión.
4. Simular la **pérdida de paquetes** en redes industriales mediante la inyección controlada de valores nulos (`NaN`).
5. Desarrollar e ilustrar gráficamente tres estrategias clásicas de imputación temporal (*Zero-imputation*, *Mean-imputation*, y *Forward-fill*).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual de seaborn
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "sans-serif"
print("Librerías importadas correctamente.")

## 📈 1. Definición y Simulación de Señales Físicas

Simularemos un entorno industrial real con tres sensores de telemetría:
- **Sensor 1: Temperatura (Celsius)**. Comportamiento base estable en torno a $70^\circ C$ con fluctuaciones cíclicas lentas de $\pm 2^\circ C$.
- **Sensor 2: Vibración (G)**. Comportamiento base oscilatorio rápido de alta frecuencia en torno a $1.0G$ con desviaciones de $\pm 0.3G$.
- **Sensor 3: Presión (Bar)**. Comportamiento base altamente estable en torno a $2.0 Bar$ con variaciones cíclicas de $\pm 0.1 Bar$.

### ⏱️ Irregularidad Temporal (Asincronía)
Para simular retrasos de red y transmisión irregular, los intervalos de muestreo (pasos temporales entre lecturas, $\Delta t$) se muestrearán de una distribución exponencial:
$$\Delta t \sim \text{Exponencial}(\lambda = 1.25) + 0.2\text{ s}$$

In [ ]:
def generate_iiot_sequence(seq_len=50, anomaly_type=None, noise_level=0.1, missing_rate=0.0):
    """
    Genera una secuencia temporal sintética de sensores IIoT con marcas de tiempo irregulares y NaNs.
    
    Parámetros:
      - seq_len: longitud de la secuencia (número de lecturas).
      - anomaly_type: None (Normal), 'spikes' (vibración), 'thermal' (calor), 'leak' (presión).
      - noise_level: nivel de ruido gaussiano blanco añadido a los sensores.
      - missing_rate: fracción de lecturas perdidas (NaN) para simular pérdida de paquetes.
    """
    # 1. Generar intervalos de tiempo irregulares (Delta t)
    dt_intervals = np.random.exponential(scale=0.8, size=seq_len) + 0.2
    timestamps = np.cumsum(dt_intervals)
    
    # 2. Generar señales base sin anomalías
    temp = 70.0 + np.sin(timestamps * 0.1) * 2.0
    vib = 1.0 + np.cos(timestamps * 0.5) * 0.3
    press = 2.0 + np.sin(timestamps * 0.05) * 0.1
    
    # 3. Inyectar anomalías según el tipo de falla
    label = 0  # 0: Normal, 1: Falla
    anomaly_desc = "Operación Normal"
    
    if anomaly_type == 'spikes':
        label = 1
        anomaly_desc = "Falla: Picos transitorios de vibración mecánica"
        # Inyectar picos de vibración en la segunda mitad de la secuencia
        spike_indices = np.random.choice(range(seq_len // 2, seq_len), size=min(3, seq_len // 4), replace=False)
        vib[spike_indices] += np.random.uniform(3.0, 5.0, size=len(spike_indices))
        
    elif anomaly_type == 'thermal':
        label = 1
        anomaly_desc = "Falla: Sobrecalentamiento gradual (Fuga térmica)"
        start_idx = seq_len // 2
        # Incremento exponencial de temperatura en la segunda mitad
        thermal_growth = np.exp(np.linspace(0, 3, seq_len - start_idx)) * 4.0
        temp[start_idx:] += thermal_growth
        # La presión experimenta una inflación leve debido al aumento térmico
        press[start_idx:] += thermal_growth * 0.05
        
    elif anomaly_type == 'leak':
        label = 1
        anomaly_desc = "Falla: Caída severa de presión (Fuga de fluido)"
        leak_idx = int(seq_len * 0.7)
        # Pérdida de presión lineal progresiva
        press[leak_idx:] = press[leak_idx:] - np.linspace(0.8, 1.2, seq_len - leak_idx)
        # Inestabilidad leve en la vibración por la pérdida del caudal hidráulico
        vib[leak_idx:] += np.random.normal(0, 0.4, size=seq_len - leak_idx)
        
    # 4. Añadir ruido de medición (ruido blanco)
    temp += np.random.normal(0, noise_level * 5.0, size=seq_len)
    vib += np.random.normal(0, noise_level * 0.5, size=seq_len)
    press += np.random.normal(0, noise_level * 0.2, size=seq_len)
    
    # 5. Aplicar pérdida de datos (NaNs)
    if missing_rate > 0.0:
        temp_mask = np.random.rand(seq_len) < missing_rate
        vib_mask = np.random.rand(seq_len) < missing_rate
        press_mask = np.random.rand(seq_len) < missing_rate
        
        temp[temp_mask] = np.nan
        vib[vib_mask] = np.nan
        press[press_mask] = np.nan
        
    # 6. Empaquetar en un DataFrame
    df = pd.DataFrame({
        'timestamp': timestamps,
        'dt': dt_intervals,
        'sensor_temp': temp,
        'sensor_vib': vib,
        'sensor_press': press
    })
    
    return df, label, anomaly_desc

print("Función de generación de datos simétrica definida.")

## 🛠️ 2. Métodos de Imputación Temporal

Definiremos tres métodos clásicos de imputación para resolver los valores nulos (`NaN`):

In [ ]:
def preprocess_and_impute(df, method='forward_fill'):
    """
    Imputa NaNs en las columnas de sensores del DataFrame.
    
    Métodos:
      - 'forward_fill': Copia el último valor válido hacia adelante (ideal para tiempo real).
      - 'mean': Reemplaza con la media de la secuencia.
      - 'zero': Reemplaza con cero.
    """
    df_imputed = df.copy()
    
    defaults = {
        'sensor_temp': 70.0,
        'sensor_vib': 1.0,
        'sensor_press': 2.0
    }
    
    for col, default_val in defaults.items():
        if df_imputed[col].isnull().all():
            df_imputed[col] = default_val
        elif method == 'forward_fill':
            # ffill() propaga el último valor, bfill() resuelve si el primer valor de la secuencia era NaN
            df_imputed[col] = df_imputed[col].ffill().bfill()
        elif method == 'mean':
            mean_val = df_imputed[col].mean()
            if pd.isnull(mean_val):
                mean_val = default_val
            df_imputed[col] = df_imputed[col].fillna(mean_val)
        elif method == 'zero':
            df_imputed[col] = df_imputed[col].fillna(0.0)
            
    return df_imputed

print("Función de imputación modular definida.")

## 📊 3. Visualización y Diagnóstico Visual

Generaremos secuencias con pérdida de datos e inyección de fallas para validar gráficamente el comportamiento y comparar las lecturas reales, la imputación y los NaNs.

In [ ]:
# Configurar simulación
seq_len = 60
noise = 0.08
missing_rate = 0.20  # 20% de pérdida de datos

# Generar secuencia de fuga térmica (thermal anomaly)
df_raw, label, desc = generate_iiot_sequence(seq_len, 'thermal', noise, missing_rate)
# Aplicar imputación por Forward-fill
df_ffill = preprocess_and_impute(df_raw, 'forward_fill')
# Aplicar imputación por Cero
df_zero = preprocess_and_impute(df_raw, 'zero')

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Graficar Sensor de Temperatura
axes[0].plot(df_ffill['timestamp'], df_ffill['sensor_temp'], label='Imputado (Forward-fill)', color='#3B82F6', alpha=0.8)
axes[0].scatter(df_raw['timestamp'], df_raw['sensor_temp'], label='Lecturas Reales', color='#1E3A8A', s=30, zorder=5)
# Identificar NaNs perdidos
nan_temp = df_raw['sensor_temp'].isna()
if nan_temp.any():
    axes[0].scatter(df_raw['timestamp'][nan_temp], df_ffill['sensor_temp'][nan_temp], 
                   label='Dato Perdido (Imputado)', color='#EF4444', marker='x', s=60, zorder=6)
axes[0].set_ylabel("Temperatura (°C)")
axes[0].set_title(f"Sensor 1: Temperatura del Sistema (Falla: {desc})")
axes[0].legend(loc='upper left')

# Graficar Sensor de Vibración (Comparando F-fill vs Zero)
axes[1].plot(df_ffill['timestamp'], df_ffill['sensor_vib'], label='Forward-fill', color='#10B981', alpha=0.8)
axes[1].plot(df_zero['timestamp'], df_zero['sensor_vib'], label='Zero Imputation', color='#F59E0B', alpha=0.6, linestyle='--')
axes[1].scatter(df_raw['timestamp'], df_raw['sensor_vib'], color='#065F46', s=30, zorder=5)
axes[1].set_ylabel("Vibración (G)")
axes[1].set_title("Sensor 2: Nivel de Vibración Mecánica")
axes[1].legend(loc='upper left')

# Graficar Sensor de Presión
axes[2].plot(df_ffill['timestamp'], df_ffill['sensor_press'], label='Imputado', color='#8B5CF6', alpha=0.8)
axes[2].scatter(df_raw['timestamp'], df_raw['sensor_press'], color='#5B21B6', s=30, zorder=5)
axes[2].set_ylabel("Presión (Bar)")
axes[2].set_xlabel("Tiempo acumulado físico (segundos)")
axes[2].set_title("Sensor 3: Presión de Fluido")

plt.tight_layout()
plt.show()

## 📁 4. Guardar Dataset Simulado de Muestra

Crearemos un conjunto de datos estructurado y lo guardaremos en la carpeta `data/` para simular un lote industrial de entrenamiento.

In [ ]:
import os

# Crear carpeta data si no existe
os.makedirs('../data', exist_ok=True)

# Generar un lote de 20 secuencias (10 normales, 10 con fallas variadas)
seq_len = 5000
lote_df = []
anomaly_types = [None, 'spikes', 'thermal', 'leak']

for i in range(5):
    for anomaly in anomaly_types:
        df, label, desc = generate_iiot_sequence(seq_len, anomaly, noise_level=0.05, missing_rate=0.15)
        # Registrar etiquetas de metadatos en el dataframe
        df['sequence_id'] = i * len(anomaly_types) + anomaly_types.index(anomaly)
        df['label'] = label
        df['description'] = desc
        lote_df.append(df)

# Concatenar y guardar como csv
dataset_full = pd.concat(lote_df, ignore_index=True)
dataset_full.to_csv('../data/sensores_iiot_simulados.csv', index=False)
print(f"Dataset generado con éxito. Registros guardados: {len(dataset_full)} en 'data/sensores_iiot_simulados.csv'")